# SupportGen · 01 — Dataset Preparation
**GenAI-Powered Customer Support Intelligence using a Fine-Tuned Transformer**

This notebook turns the raw *Tobi-Bueck/customer-support-tickets* dataset into clean, leakage-free
**train / validation / test** splits that every later notebook uses.

| Step | What happens |
|---|---|
| 1 | Install dependencies, mount Drive, write the shared helper module |
| 2 | Download the dataset directly from Hugging Face |
| 3 | Exploratory data analysis (nulls, languages, labels, lengths, duplicates) |
| 4 | Cleaning (English only, valid answers, normalised text, exact de-duplication) |
| 5 | Near-duplicate grouping + group-aware stratified 80/10/10 split |
| 6 | Leakage audit, choose max token lengths, save everything to Drive |

**Runtime:** CPU is enough (`Runtime → Change runtime type → CPU`). Takes roughly 5–10 minutes.

> Every number in this notebook is computed from the downloaded data — nothing is typed in by hand.

## 0. Setup — Google Drive & project folder
Every SupportGen notebook starts with this same cell. It mounts Google Drive so that all
notebooks share one project folder (`MyDrive/SupportGen`): notebook 01 writes the cleaned
splits there, notebooks 02–06 read them. Nothing ever needs to be uploaded from your laptop.

Set `USE_DRIVE = False` only for a throw-away run — files then live in `/content` and are lost
when the Colab runtime disconnects.

In [ ]:
# ── Setup (identical in every SupportGen notebook) ──────────────────────────
USE_DRIVE = True   # False → keep everything in /content (deleted when the runtime ends)

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/SupportGen"
else:
    BASE = "/content/SupportGen"
os.makedirs(BASE, exist_ok=True)
os.environ["SUPPORTGEN_BASE"] = BASE   # read by supportgen_common.py
sys.path.insert(0, BASE)
print("Project folder:", BASE)

In [ ]:
# Colab already ships torch, transformers, accelerate, scikit-learn, pandas, matplotlib.
# These are the extras SupportGen needs (already-installed packages are skipped).
!pip -q install datasets huggingface_hub pyarrow sentencepiece

## 1. Shared helper module
The next cell writes `supportgen_common.py` into the Drive project folder. It contains things
**every** notebook must agree on — otherwise the experiment would not be reproducible:

* the **prompt template** used for training *and* inference (a model fine-tuned on one prompt format
  should be evaluated and deployed with exactly that format),
* the random seed, file paths and run names,
* text-cleaning functions,
* a batched `generate_batch()` function and `compute_gen_metrics()` (ROUGE, BLEU, BERTScore, cosine),
* `log_experiment()`, which keeps one row per model run in `experiments.csv`.

In [ ]:
%%writefile {BASE}/supportgen_common.py
# supportgen_common.py — shared helpers for the SupportGen Colab notebooks.
# Generated by 01_dataset_preparation.ipynb; imported by notebooks 02–06.
import datetime as dt
import hashlib
import html
import json
import os
import random
import re
import time
import unicodedata

import numpy as np
import pandas as pd

# ── Paths (all inside the shared Drive folder) ─────────────────────────────
BASE = os.environ.get("SUPPORTGEN_BASE", "/content/drive/MyDrive/SupportGen")
DATA_DIR = os.path.join(BASE, "data")
PLOTS_DIR = os.path.join(BASE, "results", "plots")
RESULTS_DIR = os.path.join(BASE, "results")
MODELS_DIR = os.path.join(BASE, "models")
EXPORTS_DIR = os.path.join(BASE, "exports")
CONFIG_PATH = os.path.join(DATA_DIR, "config.json")
EXPERIMENTS_CSV = os.path.join(BASE, "experiments.csv")
FINETUNED_DIR = os.path.join(MODELS_DIR, "flan-t5-supportgen")
FINETUNED_BASE_DIR = os.path.join(MODELS_DIR, "flan-t5-base-supportgen")
CLASSIFIER_DIR = os.path.join(MODELS_DIR, "classifiers")
for _d in (DATA_DIR, PLOTS_DIR, RESULTS_DIR, MODELS_DIR, EXPORTS_DIR, CLASSIFIER_DIR):
    os.makedirs(_d, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
SEED = 42
DATASET_ID = "Tobi-Bueck/customer-support-tickets"
BASE_MODEL = "google/flan-t5-small"

# Run names used for every output / metrics file and in experiments.csv
RUN_BASE = "A_base_flan-t5-small"      # Model A: pretrained FLAN-T5-small, no fine-tuning
RUN_FT = "B_ft_flan-t5-small"          # Model B: our fine-tuned FLAN-T5-small
RUN_GEMINI = "C_gemini"                # Model C: optional Gemini benchmark
RUN_FT_BASE = "D_ft_flan-t5-base"      # Model D: optional fine-tuned FLAN-T5-base

# The SAME instruction is used for training, baseline, evaluation, Gemini and the web app.
PROMPT_TEMPLATE = (
    "Generate a professional customer support response.\n"
    "Subject: {subject}\n"
    "Customer ticket: {body}"
)
SUMMARY_TEMPLATE = (
    "Summarize the following customer support ticket in one sentence.\n"
    "Subject: {subject}\n"
    "Ticket: {body}"
)
ACTION_TEMPLATE = (
    "You are helping a human customer support agent. In one short sentence, "
    "recommend the next action the agent should take for this ticket.\n"
    "Subject: {subject}\n"
    "Ticket: {body}"
)
DEFAULT_GEN_KWARGS = dict(num_beams=4, no_repeat_ngram_size=3, early_stopping=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def get_device():
    import torch
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


# ── Text cleaning ──────────────────────────────────────────────────────────
_WS = re.compile(r"\s+")


def clean_text(x):
    """Unescape HTML entities, normalise unicode, collapse whitespace. NaN/None → ''."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = html.unescape(str(x))
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\\n", " ")  # literal backslash-n sequences left by CSV export
    return _WS.sub(" ", s).strip()


def norm_key(x):
    """Hash of lower-cased alphanumeric text, used to detect exact duplicates."""
    t = re.sub(r"[^a-z0-9]+", " ", clean_text(x).lower()).strip()
    return hashlib.md5(t.encode("utf-8")).hexdigest()


def build_prompt(subject, body, template=PROMPT_TEMPLATE):
    return template.format(subject=clean_text(subject) or "(no subject)", body=clean_text(body))


# ── I/O helpers ────────────────────────────────────────────────────────────
def _require(path, hint):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — {hint}")


def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    def _default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)

    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_default)


def load_json(path, hint="run the earlier notebook first"):
    _require(path, hint)
    with open(path) as f:
        return json.load(f)


def load_config():
    return load_json(CONFIG_PATH, "run 01_dataset_preparation.ipynb first.")


def load_split(name):
    path = os.path.join(DATA_DIR, f"{name}.parquet")
    _require(path, "run 01_dataset_preparation.ipynb first.")
    return pd.read_parquet(path)


def load_eval_subset():
    """The fixed, seeded subset of the TEST split that every model is compared on."""
    ids = load_json(os.path.join(DATA_DIR, "eval_subset_ids.json"), "run 01_dataset_preparation.ipynb first.")
    test = load_split("test").set_index("uid")
    return test.loc[ids].reset_index()


def outputs_path(run_name):
    return os.path.join(RESULTS_DIR, f"outputs_{run_name}.csv")


def metrics_path(run_name):
    return os.path.join(RESULTS_DIR, f"metrics_{run_name}.json")


# ── Generation ─────────────────────────────────────────────────────────────
def generate_batch(model, tokenizer, prompts, max_input_len, max_new_tokens,
                   batch_size=16, **gen_kwargs):
    """Generate one output per prompt. Returns (outputs, per-example latency in seconds)."""
    import torch
    device = next(model.parameters()).device
    kwargs = {**DEFAULT_GEN_KWARGS, **gen_kwargs}
    outputs, latencies = [], []
    model.eval()
    for i in range(0, len(prompts), batch_size):
        chunk = list(prompts[i:i + batch_size])
        enc = tokenizer(chunk, max_length=max_input_len, truncation=True,
                        padding=True, return_tensors="pt").to(device)
        t0 = time.perf_counter()
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=max_new_tokens, **kwargs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        outputs.extend(t.strip() for t in tokenizer.batch_decode(ids, skip_special_tokens=True))
        latencies.extend([elapsed / len(chunk)] * len(chunk))
    return outputs, latencies


# ── Generation metrics ─────────────────────────────────────────────────────
def distinct_n(texts, n):
    """Share of unique n-grams among all generated n-grams (higher = less repetitive)."""
    total, unique = 0, set()
    for t in texts:
        toks = t.lower().split()
        grams = [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]
        total += len(grams)
        unique.update(grams)
    return len(unique) / total if total else 0.0


def compute_gen_metrics(preds, refs, bertscore_model="roberta-large",
                        sts_model="sentence-transformers/all-MiniLM-L6-v2",
                        batch_size=32, return_per_example=False):
    """ROUGE-1/2/L, sacreBLEU, BERTScore (raw + baseline-rescaled), embedding cosine similarity.

    ROUGE / BERTScore / cosine are on a 0–1 scale; BLEU is on sacreBLEU's 0–100 scale.
    """
    import sacrebleu
    from bert_score import score as bert_score
    from rouge_score import rouge_scorer
    from sentence_transformers import SentenceTransformer

    preds, refs = [str(p) for p in preds], [str(r) for r in refs]
    n_empty = sum(1 for p in preds if not p.strip())
    preds = [p if p.strip() else "." for p in preds]  # BERTScore cannot score empty strings
    dev = "cuda" if get_device() == "cuda" else "cpu"

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge = [scorer.score(r, p) for p, r in zip(preds, refs)]
    per = pd.DataFrame({k: [s[k].fmeasure for s in rouge] for k in ("rouge1", "rouge2", "rougeL")})

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score

    P, R, F = bert_score(preds, refs, model_type=bertscore_model, lang="en",
                         batch_size=batch_size, device=dev, verbose=False)
    _, _, F_resc = bert_score(preds, refs, model_type=bertscore_model, lang="en",
                              batch_size=batch_size, device=dev, verbose=False,
                              rescale_with_baseline=True)
    per["bertscore_f1"] = F.numpy()
    per["bertscore_f1_rescaled"] = F_resc.numpy()

    st = SentenceTransformer(sts_model, device=dev)
    e_p = st.encode(preds, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    e_r = st.encode(refs, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    per["semantic_cosine"] = (e_p * e_r).sum(axis=1)

    summary = {
        "n": len(preds),
        "rouge1": float(per.rouge1.mean()),
        "rouge2": float(per.rouge2.mean()),
        "rougeL": float(per.rougeL.mean()),
        "bleu": float(bleu),
        "bertscore_precision": float(P.mean()),
        "bertscore_recall": float(R.mean()),
        "bertscore_f1": float(F.mean()),
        "bertscore_f1_rescaled": float(F_resc.mean()),
        "semantic_cosine": float(per.semantic_cosine.mean()),
        "avg_pred_words": float(np.mean([len(p.split()) for p in preds])),
        "avg_ref_words": float(np.mean([len(r.split()) for r in refs])),
        "distinct_1": distinct_n(preds, 1),
        "distinct_2": distinct_n(preds, 2),
        "empty_predictions": n_empty,
        "bertscore_model": bertscore_model,
        "sts_model": sts_model,
    }
    return (summary, per) if return_per_example else summary


# ── Experiment log (one row per run, updated in place) ─────────────────────
EXPERIMENT_FIELDS = [
    "run_name", "model", "date", "epochs", "learning_rate", "batch_size", "grad_accum",
    "max_input_len", "max_target_len", "train_loss", "val_loss", "test_loss",
    "rouge1", "rouge2", "rougeL", "bleu", "bertscore_f1", "semantic_cosine", "notes",
]


def log_experiment(run_name, **fields):
    if os.path.exists(EXPERIMENTS_CSV):
        df = pd.read_csv(EXPERIMENTS_CSV).astype(object)
    else:
        df = pd.DataFrame(columns=EXPERIMENT_FIELDS).astype(object)
    fields = {k: v for k, v in fields.items() if v is not None}
    fields["date"] = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
    for k in fields:
        if k not in df.columns:
            df[k] = None
    if (df["run_name"] == run_name).any():
        for k, v in fields.items():
            df.loc[df["run_name"] == run_name, k] = v
    else:
        df = pd.concat([df, pd.DataFrame([{"run_name": run_name, **fields}])], ignore_index=True)
    df.to_csv(EXPERIMENTS_CSV, index=False)
    return df

In [ ]:
import importlib, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import supportgen_common as sg
importlib.reload(sg)
sg.set_seed()
pd.set_option("display.max_colwidth", 120)
STATS = {}          # everything computed here is saved to data/eda_stats.json
def savefig(name):
    path = os.path.join(sg.PLOTS_DIR, name)
    plt.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    print("saved", path)

## 2. Download the dataset from Hugging Face
The dataset repository contains several CSV files. We load it two ways:

1. `load_dataset(...)` — the official Hugging Face view (all CSVs merged into one `train` split).
2. Each CSV separately via `hf_hub_download`, tagging rows with `source_file`.
   This lets us check whether the files **overlap** — if the same ticket appears in two files,
   a naive random split could put one copy in train and the other in test (data leakage).

In [ ]:
from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download

hf = load_dataset(sg.DATASET_ID, split="train")
print(hf)

csv_files = sorted(f for f in list_repo_files(sg.DATASET_ID, repo_type="dataset") if f.endswith(".csv"))
frames = []
for f in csv_files:
    d = pd.read_csv(hf_hub_download(sg.DATASET_ID, f, repo_type="dataset"))
    d["source_file"] = f
    print(f"{f}: {len(d):,} rows | columns: {list(d.columns)}")
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)

REQUIRED = ["subject", "body", "answer", "queue", "priority", "type", "language"]
missing = [c for c in REQUIRED if c not in raw.columns]
assert not missing, f"Dataset schema changed — missing columns {missing}"
print(f"\nRows from per-file loading: {len(raw):,} | rows from load_dataset: {len(hf):,}")
STATS["raw_rows"] = len(raw)
STATS["load_dataset_rows"] = len(hf)
STATS["files"] = {f: int((raw.source_file == f).sum()) for f in csv_files}
raw.head(3)

## 3. Exploratory Data Analysis
### 3.1 Missing values
A training example for response generation needs both a customer message (`body`) and an agent
reply (`answer`). Rows without an answer cannot be used to train the generator.

In [ ]:
null_tbl = pd.DataFrame({"nulls": raw[REQUIRED].isna().sum(),
                         "null_%": (raw[REQUIRED].isna().mean() * 100).round(2)})
display(null_tbl)
print("Nulls per source file:")
display(raw.groupby("source_file")[REQUIRED].apply(lambda g: g.isna().sum()))
STATS["nulls"] = null_tbl["nulls"].to_dict()

### 3.2 Languages and label distributions
The dataset is bilingual (English + German). FLAN-T5 was instruction-tuned mostly on English, and
evaluation metrics like BERTScore are language-specific, so **we keep English only** for this project
(multilingual support is listed as future work).

In [ ]:
display(pd.crosstab(raw.source_file, raw.language, margins=True))
STATS["language_counts"] = raw.language.value_counts().to_dict()

en_raw = raw[raw.language == "en"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ["queue", "priority", "type"]):
    vc = en_raw[col].value_counts(dropna=False).head(25)
    vc.plot.barh(ax=ax); ax.invert_yaxis(); ax.set_title(f"{col} (English, top 25)")
    STATS[f"en_{col}_counts_raw"] = {str(k): int(v) for k, v in en_raw[col].value_counts(dropna=False).items()}
plt.tight_layout(); savefig("label_distributions_raw.png")
print("Distinct queues (English):", en_raw.queue.nunique())

### 3.3 Text lengths (characters and tokens)
Transformers have a fixed maximum sequence length, so we measure lengths in **FLAN-T5 tokens**
(SentencePiece sub-words), not just characters. These percentiles decide `max_input_len` and
`max_target_len` later: we want to cover ~95 % of examples without wasting GPU memory on padding.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(sg.BASE_MODEL)

en_valid = en_raw.dropna(subset=["body", "answer"]).copy()
en_valid["prompt"] = [sg.build_prompt(s, b) for s, b in zip(en_valid.subject, en_valid.body)]
lengths = pd.DataFrame({
    "subject_chars": en_valid.subject.fillna("").str.len(),
    "body_chars": en_valid.body.str.len(),
    "answer_chars": en_valid.answer.str.len(),
    "prompt_tokens": [len(x) for x in tok(en_valid.prompt.tolist())["input_ids"]],
    "answer_tokens": [len(x) for x in tok(en_valid.answer.astype(str).tolist())["input_ids"]],
})
pct = lengths.describe(percentiles=[.5, .9, .95, .99]).T.round(1)
display(pct)
STATS["length_percentiles_en"] = pct.to_dict(orient="index")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["prompt_tokens", "answer_tokens"]):
    ax.hist(lengths[col], bins=60)
    ax.axvline(lengths[col].quantile(.95), color="red", ls="--", label="95th percentile")
    ax.set_title(col); ax.legend()
savefig("token_lengths.png")

print("Literal '\\n' sequences in body:", int(en_valid.body.str.contains(r"\\n", regex=False).sum()))

### 3.4 Duplicates and overlap between files
* **Exact duplicate bodies** — the same customer message appearing more than once.
* **Same body, different answers** — conflicting targets for the same input.
* **Cross-file overlap** — whether the separate CSV files contain the same tickets.

Normalisation for matching: lower-case, keep only letters/digits, collapse whitespace, then hash.

In [ ]:
en_valid["body_key"] = en_valid.body.map(sg.norm_key)
en_valid["pair_key"] = (en_valid.body.astype(str) + " || " + en_valid.answer.astype(str)).map(sg.norm_key)

answers_per_body = en_valid.groupby("body_key").pair_key.nunique()
dup = {
    "rows": len(en_valid),
    "unique_bodies": int(en_valid.body_key.nunique()),
    "duplicate_body_rows": int(en_valid.body_key.duplicated().sum()),
    "duplicate_body_answer_rows": int(en_valid.pair_key.duplicated().sum()),
    "bodies_with_conflicting_answers": int((answers_per_body > 1).sum()),
}
display(pd.Series(dup, name="English rows with body+answer"))
STATS["duplicates_en"] = dup

keys = {f: set(en_valid.loc[en_valid.source_file == f, "body_key"]) for f in csv_files}
overlap = pd.DataFrame([[len(keys[a] & keys[b]) for b in csv_files] for a in csv_files],
                       index=csv_files, columns=csv_files)
print("Shared unique English bodies between files (diagonal = unique bodies in that file):")
display(overlap)
STATS["cross_file_overlap_en"] = overlap.to_dict()

## 4. Cleaning
Each rule is applied in order and the remaining row count is logged, so the report can show exactly
how the final dataset was obtained.

1. Keep English tickets.
2. Normalise text (HTML entities, unicode, whitespace); missing subjects become empty strings.
3. Require a non-empty body and answer, each at least `MIN_CHARS` characters.
4. Drop exact duplicate bodies (keep the first occurrence) — removes cross-file copies and
   conflicting targets for the same input.

In [ ]:
MIN_CHARS = 20
log = []
def step(name, d):
    log.append({"step": name, "rows": len(d)}); print(f"{name:<55} {len(d):>7,}")

df = raw.copy();                                   step("raw (all files, all languages)", df)
df = df[df.language == "en"].copy();               step("English only", df)
for c in ["subject", "body", "answer"]:
    df[c] = df[c].map(sg.clean_text)
df = df[(df.body.str.len() > 0) & (df.answer.str.len() > 0)]; step("non-empty body and answer", df)
df = df[(df.body.str.len() >= MIN_CHARS) & (df.answer.str.len() >= MIN_CHARS)]
step(f"body and answer >= {MIN_CHARS} chars", df)
df["body_key"] = df.body.map(sg.norm_key)
df = df.drop_duplicates("body_key", keep="first"); step("drop exact duplicate bodies", df)

tag_cols = [c for c in df.columns if c.startswith("tag_")]
df["tags"] = df[tag_cols].apply(lambda r: "; ".join(str(t) for t in r if pd.notna(t) and str(t).strip()), axis=1)
df["uid"] = df.body_key.str[:16]
assert df.uid.is_unique
STATS["cleaning_log"] = log

### 4.1 Queue label policy (for the queue classifier only)
The English data contains a few large support queues and a long tail of small queues.
A classifier cannot learn a class from a handful of examples, so queues with fewer than
`MIN_QUEUE_ROWS` cleaned English tickets are merged into **`Other`** in a new column `queue_group`.
The original `queue` column is kept untouched. This affects only the classifier — the response
generator is trained on **all** cleaned tickets.

In [ ]:
MIN_QUEUE_ROWS = 200
qc = df.queue.value_counts()
keep = qc[qc >= MIN_QUEUE_ROWS].index.tolist()
df["queue_group"] = np.where(df.queue.isin(keep), df.queue, "Other")
print(f"{len(keep)} queues kept, {qc.size - len(keep)} merged into 'Other'")
display(df.queue_group.value_counts().to_frame("rows"))
STATS["queue_groups"] = df.queue_group.value_counts().to_dict()

## 5. Leakage-safe train / validation / test split
**Why not a plain random split?** Customer-support data often contains near-identical tickets
("I was charged twice…" with a different name). If one copy is in train and its twin in test, the
test score measures memorisation, not generalisation.

**Approach**
1. Represent each ticket (subject + body) as a TF-IDF vector.
2. Link every pair of tickets with cosine similarity ≥ `NEAR_DUP_COS` (nearest-neighbour search).
3. Connected components of that graph become **groups**; a group is never split across sets.
4. `StratifiedGroupKFold` (10 folds) → fold 0 = test, fold 1 = validation, folds 2–9 = train.
   That gives ≈ 80 / 10 / 10 while keeping the `queue_group` distribution similar in every split.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

NEAR_DUP_COS, K = 0.90, 10
df = df.reset_index(drop=True)
texts = (df.subject + " " + df.body).tolist()
X = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
dist, idx = NearestNeighbors(n_neighbors=K, metric="cosine", algorithm="brute", n_jobs=-1).fit(X).kneighbors(X)

n = len(df)
src = np.repeat(np.arange(n), K)
mask = ((1 - dist.ravel()) >= NEAR_DUP_COS) & (idx.ravel() != src)
graph = coo_matrix((np.ones(mask.sum()), (src[mask], idx.ravel()[mask])), shape=(n, n))
n_groups, df["group_id"] = connected_components(graph, directed=False)
gsize = df.group_id.value_counts()
STATS["near_duplicates"] = {"threshold_cosine": NEAR_DUP_COS, "groups": int(n_groups),
                            "tickets_in_multi_ticket_groups": int(gsize[gsize > 1].sum()),
                            "largest_group": int(gsize.max())}
print(STATS["near_duplicates"])

# show one near-duplicate pair as evidence
multi = gsize[gsize > 1]
if len(multi):
    display(df[df.group_id == multi.index[0]][["subject", "body"]].head(2))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
folds = list(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=sg.SEED)
             .split(df, df.queue_group, groups=df.group_id))
test_idx, val_idx = folds[0][1], folds[1][1]
df["split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

# no group may appear in more than one split
assert df.groupby("group_id").split.nunique().max() == 1
print(df.split.value_counts())

### 5.1 Leakage audit
Independent check: fit TF-IDF on **train only**, then find each validation/test ticket's most
similar training ticket. Any val/test ticket still above the threshold is removed.

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform([texts[i] for i in df.index[df.split == "train"]])
nn_tr = NearestNeighbors(n_neighbors=1, metric="cosine", algorithm="brute", n_jobs=-1).fit(Xtr)
audit = {}
for s in ["val", "test"]:
    ids = df.index[df.split == s]
    d, _ = nn_tr.kneighbors(vec.transform([texts[i] for i in ids]))
    sim = 1 - d.ravel()
    flagged = ids[sim >= NEAR_DUP_COS]
    df = df.drop(index=flagged)
    audit[s] = {"max_similarity_to_train": float(sim.max()),
                "mean_similarity_to_train": float(sim.mean()),
                "removed_near_duplicates": int(len(flagged))}
df = df.reset_index(drop=True)
display(pd.DataFrame(audit).T)
STATS["leakage_audit"] = audit

In [ ]:
split_sizes = df.split.value_counts().reindex(["train", "val", "test"])
display(split_sizes.to_frame("rows").assign(pct=lambda t: (t.rows / t.rows.sum() * 100).round(1)))
for col in ["queue_group", "priority", "type"]:
    print(f"\n{col} distribution per split (%):")
    display((pd.crosstab(df[col].fillna("<missing>"), df.split, normalize="columns") * 100).round(1)[["train", "val", "test"]])
STATS["split_sizes"] = split_sizes.to_dict()

## 6. Choose maximum sequence lengths
Inputs/targets longer than the maximum are truncated. We take the 95th percentile of
**training-set** token lengths, round up to a multiple of 32 (GPU-friendly), and clamp to sensible
bounds. The share of truncated examples is reported so the choice can be justified in the report.

In [ ]:
train_df = df[df.split == "train"]
p_tok = np.array([len(x) for x in tok([sg.build_prompt(s, b) for s, b in zip(train_df.subject, train_df.body)])["input_ids"]])
a_tok = np.array([len(x) for x in tok(train_df.answer.tolist())["input_ids"]])
round32 = lambda v: int(np.ceil(v / 32) * 32)
MAX_INPUT_LEN = int(np.clip(round32(np.percentile(p_tok, 95)), 128, 512))
MAX_TARGET_LEN = int(np.clip(round32(np.percentile(a_tok, 95)), 64, 512))
STATS["max_lengths"] = {
    "max_input_len": MAX_INPUT_LEN, "max_target_len": MAX_TARGET_LEN,
    "pct_inputs_truncated": float((p_tok > MAX_INPUT_LEN).mean() * 100),
    "pct_targets_truncated": float((a_tok > MAX_TARGET_LEN).mean() * 100),
}
print(STATS["max_lengths"])

## 7. Save splits, evaluation subset and configuration
* `train/val/test.parquet` — the splits used by every later notebook.
* `eval_subset_ids.json` — a fixed, seeded sample of test tickets. Base FLAN-T5, our fine-tuned
  model and Gemini are all compared on **exactly these tickets**.
* `config.json` — prompt template, max lengths and split metadata (copied into the trained model folder
  later so the web app uses the identical prompt).
* `eda_stats.json` — all statistics above, for the report.

In [ ]:
N_EVAL = 500
cols = ["uid", "subject", "body", "answer", "queue", "queue_group", "priority", "type", "tags", "source_file", "group_id"]
for s in ["train", "val", "test"]:
    df.loc[df.split == s, cols].reset_index(drop=True).to_parquet(os.path.join(sg.DATA_DIR, f"{s}.parquet"), index=False)

test_uids = df.loc[df.split == "test", "uid"]
eval_ids = test_uids.sample(n=min(N_EVAL, len(test_uids)), random_state=sg.SEED).tolist()
sg.save_json(eval_ids, os.path.join(sg.DATA_DIR, "eval_subset_ids.json"))

config = {
    "dataset_id": sg.DATASET_ID, "seed": sg.SEED, "language": "en",
    "prompt_template": sg.PROMPT_TEMPLATE, "summary_template": sg.SUMMARY_TEMPLATE,
    "action_template": sg.ACTION_TEMPLATE,
    "max_input_len": MAX_INPUT_LEN, "max_target_len": MAX_TARGET_LEN,
    "min_chars": MIN_CHARS, "min_queue_rows": MIN_QUEUE_ROWS, "near_dup_cosine": NEAR_DUP_COS,
    "queue_groups": sorted(df.queue_group.unique().tolist()),
    "split_sizes": STATS["split_sizes"], "eval_subset_size": len(eval_ids),
}
sg.save_json(config, sg.CONFIG_PATH)
sg.save_json(STATS["split_sizes"], os.path.join(sg.DATA_DIR, "split_sizes.json"))
sg.save_json(STATS, os.path.join(sg.DATA_DIR, "eda_stats.json"))
print("Saved to", sg.DATA_DIR, "→", sorted(os.listdir(sg.DATA_DIR)))
print(json.dumps(config, indent=2))

## Summary
Record the printed numbers (split sizes, cleaning log, max lengths, near-duplicate statistics) in the
report's *Dataset* and *Pre-processing* chapters — they are all stored in `data/eda_stats.json`.

**Dataset licence & caveat:** *Tobi-Bueck/customer-support-tickets* is released under **CC BY-NC 4.0**
(non-commercial use with attribution). According to its dataset card it is **synthetically generated**,
so results reflect performance on synthetic support conversations, not a specific real company.

➡️ **Next:** `02_baseline_flan_t5.ipynb` (switch the runtime to **GPU**).